In [1]:
import sys
sys.path.append("..")

import h5py
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from utils.das_dataset import DasDataset
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from utils.noise_scheduler import LinearNoiseScheduler 
from IPython.display import clear_output

In [2]:
das_data = Path("/datasets/training/earthquake/share/SISSLE_south30")
train_files_noise = [das_data/"train_1000.h5"]#sorted(das_data.glob("train_*.h5"))
test_files_noise = [das_data/"test_1000.h5"]#sorted(das_data.glob("test_*.h5"))

device= "cuda" if torch.cuda.is_available() else "cpu"

train_set_noise = DasDataset(train_files_noise)
test_set_noise = DasDataset(test_files_noise)

In [ ]:
tail_scores_path = Path ("data/noise_tail_scores.npy")
tails_scores_path.parent.mkdir(exist_ok = True)

if tail_scores_path.exists():
    tail_scores = np.load(tail_scores_path)

else:
    tail_scores = []
    for file, file_path in enumerate (train_files_noise):
        print(f"processing file {file+1} / {len(train_files_noise)} :") 
         f = h5py.File(file_path, "r")
         patches = f["patches"]
         num_patches = patches.shape[0]
         for index in tqdm(range(num_patches)):
            patch = np.array(patches[index])
            mu = np.median(patch)
            centered_patch = patch - mu
            tail_score = np.percentile(np.abs(centered_patch),99.9)
            tail_scores.append(tail_score)

    tail_scores = np.asarray(tail_scores, dtype = np.float32)
    np.save(tail_scores_path, tail_scores)

    print("Tail scores saved")
print("Tail scores shape:", tail_scores.shape)            